In [0]:
declare or replace variable smart_on_fhir_model STRING;
-- set variable smart_on_fhir_model = 'dev_matthew_giglia_epic_on_fhir_requests';
set variable smart_on_fhir_model = 'sandbox_epic_on_fhir_requests';

SELECT
	http_method
	, resource
	, action
	, result.response_status_code
	, result.response_time_seconds
	, result.response_url
	, result.response_text::variant response_text_variant
	,ai_query(
		endpoint => "databricks-gemini-2-5-flash"
		,request => CONCAT("summarize the patient demographics from ", result.response_text)
	 ) as patient_demographic_summary
FROM (
	SELECT
		http_method
		, resource
		, action
		, ai_query(smart_on_fhir_model,
			request => named_struct(
				'http_method', http_method
				, 'resource', resource
				, 'action', action
				, 'data', data
			)
			, returnType => 'STRUCT<response_headers: STRING, response_url: STRING, response_time_seconds: DOUBLE, response_status_code: INT, response_text: STRING>'
		) as result
	FROM VALUES
		('get', 'Patient', 'T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB', CAST(null AS STRING))
		AS requests(http_method, resource, action, data)
);
